# Flujo de cadena secuencial

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableSequence, RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
import os

c:\Users\eparedes\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
with open("api_key.txt") as archivo:
  apikey = archivo.read()
os.environ["OPENAI_API_KEY"] = apikey


In [3]:
# Inicializar el modelo de chat
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, verbose=True)

In [4]:
# Prompt para traducción
translate_prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un traductor profesional que traduce reseñas al español con precisión y naturalidad."),
    ("user", "Traduce la siguiente reseña al español:\n{review}")
])

# Prompt para resumen
summarize_prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente que resume textos de manera clara y concisa en una sola frase."),
    ("user", "Resume la siguiente reseña en una frase:\n{Spanish_rev}")
])

# Prompt para detectar idioma
language_prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un experto en lingüística que identifica el idioma de un texto."),
    ("user", "¿Indica en una sola palabra en qué idioma está escrita la siguiente reseña?:\n{review}")
])

# Prompt para respuesta de seguimiento
followup_prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente que escribe respuestas de seguimiento en un tono profesional y amigable."),
    ("user", "Escribe una respuesta de seguimiento al siguiente resumen en el idioma especificado:\n\nResumen: {resumen}\n\nIdioma: {lenguaje}")
])

In [5]:
# Crear las cadenas individuales usando LCEL
translate_chain = translate_prompt | llm | StrOutputParser()
summarize_chain = summarize_prompt | llm | StrOutputParser()
language_chain = language_prompt | llm | StrOutputParser()
followup_chain = followup_prompt | llm | StrOutputParser()

In [6]:
#probamos una cadena
summarize_chain.invoke({"Spanish_rev": "Muy buenos días, espero que se encuentre bien de salud."})

'El remitente desea buenos días y espera que el destinatario esté bien de salud.'

In [7]:
# Combinar las cadenas en una secuencia
complet_chain = RunnableSequence(
    # Paso 1: Ejecutar traducción, idioma y pasar la reseña original en paralelo
    RunnableParallel({
        "Spanish_rev": translate_chain,
        "lenguaje": language_chain
    }),
    # Paso 2: Generar el resumen usando Spanish_rev y combinar con los otros resultados
    RunnableParallel({
        "resumen": lambda x: summarize_chain.invoke({"Spanish_rev": x["Spanish_rev"]}),
        "lenguaje": lambda x: x["lenguaje"]
    }),
    # Paso 3: Generar el mensaje de seguimiento usando resumen y lenguaje(ingles)
    lambda x: {
        "mensaje_seg": followup_chain.invoke({
            "resumen": x["resumen"],
            "lenguaje": x["lenguaje"]
        })
    }
)

In [8]:
review = """I recently purchased the Smartphone Alcatel, and it has exceeded my expectations in every way.
The battery life is incredible, lasting me two full days with heavy use.
The camera quality is outstanding, capturing crisp and vibrant photos even in low light.
The sleek design and lightweight build make it a pleasure to use daily.
Additionally, the performance is top-notch, handling multiple apps and tasks without any lag.
I highly recommend the Smartphone Alcatel to anyone looking for a reliable and high-performing device."""

In [10]:
review2 = """Recientemente compré el smartphone Alcatel y ha superado mis expectativas en todos los sentidos.
La duración de la batería es increíble, me dura dos días completos con un uso intensivo.
La calidad de la cámara es excepcional, capturando fotos nítidas y vibrantes incluso con poca luz.
Su diseño elegante y su estructura ligera hacen que sea un placer usarlo a diario.
Además, el rendimiento es de primera clase, manejando múltiples aplicaciones y tareas sin ningún tipo de retraso.
Recomiendo encarecidamente el smartphone Alcatel a cualquiera que busque un dispositivo confiable y de alto rendimiento."""

In [9]:
# Ejecutar la cadena
import pprint
result = complet_chain.invoke({"review": review})
print(result)

{'mensaje_seg': "Subject: Follow-Up on Alcatel Smartphone Review\n\nDear [Recipient's Name],\n\nThank you for sharing your insightful summary of the Alcatel smartphone. I’m glad to hear that it has exceeded expectations in terms of battery life, camera quality, design, and performance. \n\nIf you have any further thoughts or experiences with the device, I would love to hear more about them. Additionally, if you have any specific features that stood out to you or any comparisons with other smartphones, please feel free to share.\n\nLooking forward to your response!\n\nBest regards,\n\n[Your Name]  \n[Your Position]  \n[Your Contact Information]  "}


In [11]:
# Ejecutar la cadena
import pprint
result = complet_chain.invoke({"review": review2})
print(result)

{'mensaje_seg': '¡Hola!\n\nGracias por compartir tu resumen sobre el smartphone Alcatel. Me alegra saber que ha superado tus expectativas en aspectos tan importantes como la duración de la batería, la calidad de la cámara y el diseño. Sin duda, un dispositivo que combina rendimiento y estética es muy atractivo para los usuarios. \n\nSi tienes alguna experiencia adicional o características específicas que te gustaría destacar, no dudes en compartirlas. ¡Estoy aquí para ayudarte!\n\nSaludos cordiales.'}


--- ejemplo 2 ---

In [12]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableSequence, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
import pprint

# Configuración de la API Key
with open("api_key.txt") as archivo:
    apikey = archivo.read().strip()
os.environ["OPENAI_API_KEY"] = apikey

# Inicializar el modelo
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# ==========================================
# 1. DEFINICIÓN DE PROMPTS (Componentes Atómicos)
# ==========================================

# Detector de toxicidad
toxicity_prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un moderador de contenido. Analiza el comentario y responde ESTRICTAMENTE con una palabra: 'TOXICO' o 'SEGURO'."),
    ("user", "Analiza este comentario: {comentario}")
])

# Extractor de tópicos/palabras clave
topics_prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un extractor de datos. Extrae las 3 palabras clave principales del texto separadas por comas."),
    ("user", "Extrae los tópicos de: {comentario}")
])

# Clasificador final de estado
policy_prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres el sistema central de políticas. Si la toxicidad es 'TOXICO', el estado es 'RECHAZADO'. Si es 'SEGURO', el estado es 'APROBADO'."),
    ("user", "Determina el estado final. Toxicidad: {toxicidad}")
])

# Generador de justificación/notificación
alert_prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un bot de soporte técnico corporativo. Escribe una notificación formal indicando el estado del comentario y las palabras clave asociadas."),
    ("user", "Genera la alerta con estos datos:\nEstado: {estado}\nPalabras clave: {palabras_clave}")
])

# ==========================================
# 2. CREACIÓN DE CADENAS INDIVIDUALES
# ==========================================
toxicity_chain = toxicity_prompt | llm | StrOutputParser()
topics_chain = topics_prompt | llm | StrOutputParser()
policy_chain = policy_prompt | llm | StrOutputParser()
alert_chain = alert_prompt | llm | StrOutputParser()

# ==========================================
# 3. ORQUESTACIÓN DE LA ARQUITECTURA (Para explicar en clase)
# ==========================================
moderator_pipeline = RunnableSequence(
    
    # PASO 1: Análisis simultáneo del input original
    RunnableParallel({
        "toxicidad": toxicity_chain,
        "palabras_clave": topics_chain
    }),
    
    # PASO 2: Tomar el resultado de toxicidad para evaluar la política de aprobación
    RunnableParallel({
        "estado": lambda x: policy_chain.invoke({"toxicidad": x["toxicidad"]}),
        "palabras_clave": lambda x: x["palabras_clave"]
    }),
    
    # PASO 3: Generar la salida final unificada
    lambda x: {
        "notificacion_soporte": alert_chain.invoke({
            "estado": x["estado"],
            "palabras_clave": x["palabras_clave"]
        })
    }
)



In [13]:
# ==========================================
# 4. PRUEBA CON DOS ESCENARIOS DIDÁCTICOS
# ==========================================

comentario_alumno_1 = "El servicio de atención al cliente es una basura, odio a todos los que trabajan ahí."

print("--- EVALUANDO COMENTARIO 1 ---")
resultado_1 = moderator_pipeline.invoke({"comentario": comentario_alumno_1})
print(resultado_1["notificacion_soporte"])



--- EVALUANDO COMENTARIO 1 ---
**Notificación de Estado de Comentario**

Estimado usuario,

Le informamos que el estado final de su comentario ha sido actualizado a **RECHAZADO**. 

Palabras clave asociadas: **servicio, atención, cliente**.

Si tiene alguna pregunta o requiere asistencia adicional, no dude en ponerse en contacto con nuestro equipo de soporte técnico.

Atentamente,  
[Nombre del Departamento de Soporte Técnico]  
[Nombre de la Empresa]  
[Información de Contacto]  


In [14]:
comentario_alumno_2 = "Excelente producto, la pantalla del celular brilla genial y el envío llegó rápido."
print("\n--- EVALUANDO COMENTARIO 2 ---")
resultado_2 = moderator_pipeline.invoke({"comentario": comentario_alumno_2})
print(resultado_2["notificacion_soporte"])




--- EVALUANDO COMENTARIO 2 ---
**Notificación de Estado de Comentario**

Estimado usuario,

Nos complace informarle que el estado final de su comentario ha sido **APROBADO**. Agradecemos su participación y el tiempo dedicado a compartir su opinión.

Palabras clave asociadas a su comentario: **producto, pantalla, envío**.

Si tiene alguna pregunta adicional o requiere más información, no dude en ponerse en contacto con nuestro equipo de soporte técnico.

Atentamente,

[Su Nombre]  
[Su Cargo]  
[Nombre de la Empresa]  
[Información de Contacto]  


--- con ollama ---